# Measures of dispersion
Range, variance, mean absolute deviation, standard deviation and coefficient of variation, on small numbers and on the Titanic data, plus the $n - 1$ experiment.

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# the five values used throughout the Note
x = np.array([3, 2, 1, 5, 4])

## 1. The five measures on 3, 2, 1, 5, 4

In [ ]:
# range: largest minus smallest
print('range', x.max() - x.min())
# variance: population (divide by n) and sample (divide by n - 1)
print('population variance', np.var(x), ' sample variance', np.var(x, ddof=1))
# the distances from the mean always add up to 0
print('sum of distances', (x - x.mean()).sum())
# mean absolute deviation: average distance without the sign
print('mean absolute deviation', np.abs(x - x.mean()).mean())
# standard deviation: square root of the variance
print('standard deviation', np.std(x))
# coefficient of variation, in percent
print('CV %', np.std(x) / x.mean() * 100)

## 2. One outlier

In [ ]:
# add a far value and watch every measure grow
x2 = np.append(x, 50)
for name, f in [('range', np.ptp), ('variance', np.var), ('std', np.std),
                ('MAD', lambda v: np.abs(v - v.mean()).mean())]:
    print(f'{name:9s} before {f(x):7.2f}   after {f(x2):7.2f}')

## 3. Units: variance against standard deviation

In [ ]:
# four salaries in LPA (lakh rupees per annum)
lpa = np.array([16, 17, 13, 14])
print('variance', np.var(lpa), 'LPA squared')
print('std', round(np.std(lpa), 2), 'LPA')

## 4. Coefficient of variation on the Titanic

In [ ]:
df = pd.read_csv('data/titanic_train.csv')
# pandas' std divides by n - 1, like a spreadsheet's STDEV
for col in ['Age', 'Fare']:
    m, s = df[col].mean(), df[col].std()
    print(f'{col}: mean {m:.2f}, std {s:.2f}, CV {s / m * 100:.1f}%')

## 5. Why divide by n - 1
First the exact count: every sample of two values from the population 1, 2, 3, 4.

In [ ]:
import itertools

pop = np.array([1, 2, 3, 4])
print('true variance', pop.var())
# all 16 samples of size 2, drawn with replacement
samples = np.array(list(itertools.product(pop, repeat=2)))
print('average, dividing by n    ', samples.var(axis=1, ddof=0).mean())
print('average, dividing by n - 1', samples.var(axis=1, ddof=1).mean())

Now the same experiment on the 714 known Titanic ages, treated as a population. Change `sizes` or the number of samples and rerun.

In [ ]:
ages = df['Age'].dropna().to_numpy()
rng = np.random.default_rng(0)          # fixed seed: same numbers every run
sizes = np.arange(2, 21)
by_n = [rng.choice(ages, (20_000, n)).var(axis=1, ddof=0).mean() for n in sizes]
by_n1 = [rng.choice(ages, (20_000, n)).var(axis=1, ddof=1).mean() for n in sizes]

fig = go.Figure()
fig.add_hline(y=ages.var(), line_dash='dash', annotation_text='true variance')
fig.add_scatter(x=sizes, y=by_n1, name='divide by n - 1')
fig.add_scatter(x=sizes, y=by_n, name='divide by n')
fig.update_layout(xaxis_title='sample size n', yaxis_title='average sample variance')
fig.show()

## 6. Checking the identity behind n - 1
For any point $\mu$: sum of $(x - \mu)^2$ = sum of $(x - \bar{x})^2$ + $n(\bar{x} - \mu)^2$.

In [ ]:
mu = 2.5
left = ((x - mu) ** 2).sum()
right = ((x - x.mean()) ** 2).sum() + len(x) * (x.mean() - mu) ** 2
print(left, right)

## 7. Variance around a point v
For two random samples of five ages, the average squared distance to v is smallest at the sample mean, and larger at the population mean (Figure 5).

In [ ]:
mu = ages.mean()
grid = np.linspace(12, 56, 4401)
for seed in (0, 1):
    s5 = np.random.default_rng(seed).choice(ages, 5, replace=False)
    around = lambda v: ((s5 - v) ** 2).mean()          # divide by n
    v_best = grid[np.argmin([around(v) for v in grid])]
    print(s5, 'sample mean', s5.mean().round(1), 'bottom of U at', v_best.round(2),
          'around x-bar', round(around(s5.mean()), 1), 'around mu', round(around(mu), 1))

## 8. Dividing by n gives (n - 1)/n of the true variance

In [ ]:
ratio = np.array(by_n) / ages.var()
for n, r in zip(sizes[:4], ratio[:4]):
    print(n, round(r, 3), 'pattern (n-1)/n =', round((n - 1) / n, 3))